# Task 2 — Review Clustering

## Goal

Discover recurring topics in customer reviews without using predefined labels.

Unlike sentiment classification, clustering is an unsupervised learning task.

We will:

- use `reviews.text` as the clustering input
- convert the text into TF-IDF features
- test KMeans with K = 4, 5 and 6
- inspect keywords and example reviews
- choose the most interpretable clustering solution
- assign human-readable names to the final clusters

The columns `name`, `categories`, `reviews.rating`, and `sentiment` will not be used to train KMeans. They will only help us interpret the clusters afterward.

## Daily Recap

### Previous work

Task 1 — Sentiment Analysis is complete.

The selected sentiment classifier is:

**Balanced Linear SVM + TF-IDF**

It achieved the strongest balanced performance across Negative, Neutral and Positive reviews, with Macro F1 = 0.56.

### Today

We start Task 2 — Clustering.

Current decision:

- clustering input: `reviews.text`
- feature representation: TF-IDF
- initial feature limit: 5,000
- algorithm: KMeans
- test K = 4, 5 and 6
- choose the solution based primarily on semantic coherence and interpretability

## 1. Load and validate the cleaned dataset

### Question

Are we starting clustering from the same cleaned dataset used in sentiment analysis?

### What we did

We load the Day 1 cleaned dataset and verify that the review text required for clustering exists.

In [1]:
# Import the libraries needed for this clustering task.
from pathlib import Path

import pandas as pd

# Find the project root folder.
PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is running from the "notebooks" folder,
# move one level up to the project root.
if not (PROJECT_ROOT / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Build the path to the cleaned dataset.
CLEAN_DATA_PATH = PROJECT_ROOT / "datasets" / "clean_reviews.csv"

# Load the cleaned dataset.
df = pd.read_csv(
    CLEAN_DATA_PATH,
    low_memory=False
)

# Display the dataset dimensions.
print("Dataset shape:", df.shape)

# Check that the review-text column exists.
assert "reviews.text" in df.columns

# Check that review text has no missing values.
assert df["reviews.text"].isna().sum() == 0

print("Clustering data validation passed.")

Dataset shape: (34615, 7)
Clustering data validation passed.


In [ ]:
## 2. Select the clustering input

### Question

What information should KMeans use to create the clusters?

### Decision

We use only `reviews.text`.

We intentionally do not use:

- sentiment
- rating
- product name
- categories

Those columns could force the groups to follow information we already know.

Instead, we want the topics to emerge from the review language itself.

In [2]:
# Select only the review text for clustering.
reviews = df["reviews.text"]

# Check how many reviews will be clustered.
print("Number of reviews:", len(reviews))

# Display a few examples.
reviews.head()

Number of reviews: 34615


0    This product so far has not disappointed. My c...
1    great for beginner or experienced person. Boug...
2    Inexpensive tablet for him to use and learn on...
3    I've had my Fire HD 8 two weeks now and I love...
4    I bought this for my grand daughter when she c...
Name: reviews.text, dtype: str

## 3. Convert review text into TF-IDF features

### Question

How can KMeans measure similarity between pieces of text?

### What we did

We convert the reviews into numerical TF-IDF features.

For clustering we use:

- maximum 5,000 features
- English stop-word removal
- single words and two-word phrases

In [3]:
# Import the TF-IDF vectorizer.
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF representation for clustering.
clustering_tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

# Learn the vocabulary and convert every review into numbers.
X_cluster = clustering_tfidf.fit_transform(
    reviews
)

# Display the matrix dimensions.
print("TF-IDF clustering shape:", X_cluster.shape)

TF-IDF clustering shape: (34615, 5000)


In [ ]:
## 4. Test different numbers of clusters

### Question

How many groups should KMeans create?

### What we did

We test three reasonable starting values:

- K = 4
- K = 5
- K = 6

We do not assume that one value is correct in advance.

Later, we will inspect the topics produced by each solution and choose the most interpretable one.

In [4]:
# Import the KMeans clustering algorithm.
from sklearn.cluster import KMeans

# Store the K values we want to test.
k_values = [4, 5, 6]

# Create dictionaries where we will save
# the trained models and their cluster labels.
kmeans_models = {}
cluster_labels = {}

# Train one KMeans model for each K value.
for k in k_values:

    # Create the KMeans model.
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    # Train KMeans and assign every review to a cluster.
    labels = model.fit_predict(
        X_cluster
    )

    # Save the trained model.
    kmeans_models[k] = model

    # Save the cluster assignment for every review.
    cluster_labels[k] = labels

    # Print the size of each cluster.
    print(f"\nK = {k}")

    print(
        pd.Series(labels)
        .value_counts()
        .sort_index()
    )


K = 4
0    21248
1     6365
2     3932
3     3070
Name: count, dtype: int64

K = 5
0     3801
1     2898
2    17484
3     4514
4     5918
Name: count, dtype: int64

K = 6
0     4679
1     2706
2    15435
3     2204
4     3749
5     5842
Name: count, dtype: int64


In [19]:
# Map our numerical cluster IDs to the
# semantic names agreed by the team.
cluster_names = {
    0: "Kindle & Reading",
    1: "Ease of Use & Setup",
    2: "Fire TV & Streaming",
    3: "General & Mixed Feedback",
    4: "Kids, Family & Gifts",
    5: "Tablets & Value"
}

# Add the readable name for every review.
df["cluster_name"] = df["cluster"].map(cluster_names)

# Check the result.
df[["cluster", "cluster_name"]].head()

,cluster,cluster_name
0,2,Fire TV & Streaming
1,4,"Kids, Family & Gifts"
2,5,Tablets & Value
3,5,Tablets & Value
4,5,Tablets & Value


## 5. Inspect the top keywords in each cluster

### Question

What topic or theme does each cluster appear to represent?

### What we did

For each KMeans solution, we inspect the TF-IDF features that are closest to each cluster center.

These high-weight words and phrases help us understand what language is most representative of each cluster.

In [20]:
# Get the words and phrases learned by the TF-IDF vectorizer.
feature_names = clustering_tfidf.get_feature_names_out()

# Choose how many top terms we want to inspect per cluster.
top_n = 12

# Inspect the top terms for K = 4, 5, and 6.
for k in k_values:

    print(f"\n{'=' * 50}")
    print(f"K = {k}")
    print(f"{'=' * 50}")

    # Get the trained KMeans model for this value of K.
    model = kmeans_models[k]

    # Each row in cluster_centers_ represents one cluster.
    # Each column represents one TF-IDF feature.
    for cluster_number, center in enumerate(model.cluster_centers_):

        # Sort the feature weights from highest to lowest.
        top_indices = center.argsort()[::-1][:top_n]

        # Convert the feature positions back into words/phrases.
        top_terms = feature_names[top_indices]

        print(
            f"\nCluster {cluster_number}: "
            + ", ".join(top_terms)
        )


K = 4

Cluster 0: great, love, kindle, product, amazon, tv, works, like, echo, good, use, alexa

Cluster 1: tablet, price, great, good, great tablet, kids, great price, amazon, apps, tablet great, love, tablet price

Cluster 2: loves, bought, gift, old, year, year old, daughter, christmas, son, tablet, great, got

Cluster 3: easy, easy use, use, set, easy set, great, love, tablet, product, use great, tablet easy, good

K = 5

Cluster 0: loves, bought, gift, old, year, year old, daughter, christmas, son, tablet, great, daughter loves

Cluster 1: easy, easy use, use, set, easy set, great, love, tablet, product, use great, tablet easy, good

Cluster 2: great, kindle, love, product, tv, works, good, amazon, like, read, reading, use

Cluster 3: echo, alexa, music, love, home, great, amazon, speaker, weather, fun, questions, sound

Cluster 4: tablet, price, great, good, great tablet, kids, great price, amazon, tablet great, love, apps, tablet price

K = 6

Cluster 0: kindle, read, reading, 

In [21]:
# Add the K=6 cluster assignment
# to each review in the DataFrame.
df["cluster"] = cluster_labels[6]

# Check how many reviews belong to each cluster.
df["cluster"].value_counts().sort_index()

cluster
0     4679
1     2706
2    15435
3     2204
4     3749
5     5842
Name: count, dtype: int64

In [22]:
# Show three example reviews from each cluster.
for cluster_number in range(6):

    print(f"\n{'=' * 60}")
    print(f"CLUSTER {cluster_number}")
    print(f"{'=' * 60}")

    # Select reviews assigned to this cluster.
    cluster_reviews = df[
        df["cluster"] == cluster_number
    ]

    # Display three example review texts.
    display(
        cluster_reviews[
            ["reviews.text"]
        ].head(3)
    )


CLUSTER 0


,reviews.text
6,"Great for e-reading on the go, nice and light ..."
8,Great as a device to read books. I like that i...
9,I love ordering books and reading them with th...



CLUSTER 1


,reviews.text
7,"I gave this as a Christmas gift to my inlaws, ..."
11,"Excellent product. Easy to use, large screen m..."
12,Wanted my father to have his first tablet and ...



CLUSTER 2


...

,reviews.text
32,Works great. Love the portability of books. Ov...
36,Overall a nice product for traveling purposes ...
99,Great product for the kids gaming apps parenta...



CLUSTER 4


,reviews.text
1,great for beginner or experienced person. Boug...
30,If your looking for a affordable gift this is ...
40,Bought this for my 87 year old grandmother who...



CLUSTER 5


,reviews.text
2,Inexpensive tablet for him to use and learn on...
3,I've had my Fire HD 8 two weeks now and I love...
4,I bought this for my grand daughter when she c...


## 7. Compare clustering quality with Silhouette Score

### Question

Besides manually reading the clusters, can we use a numerical metric to compare K = 4, 5 and 6?

### What we did

We calculate the Silhouette Score for each clustering solution.

A higher score generally means that observations are more similar to their own cluster and more separated from other clusters.

Because this is a large text dataset, we calculate the score on a reproducible sample to keep the computation manageable.

In [23]:
# Import the Silhouette Score metric.
from sklearn.metrics import silhouette_score

# Create an empty dictionary to store
# the score for each value of K.
silhouette_scores = {}

# Compare K = 4, 5, and 6.
for k in k_values:

    # Calculate the silhouette score using
    # a sample of reviews to keep computation manageable.
    score = silhouette_score(
        X_cluster,
        cluster_labels[k],
        sample_size=5000,
        random_state=42
    )

    # Save the result.
    silhouette_scores[k] = score

    # Display the score.
    print(
        f"K = {k}: Silhouette Score = {score:.4f}"
    )

K = 4: Silhouette Score = 0.0054
K = 5: Silhouette Score = 0.0066
K = 6: Silhouette Score = 0.0074


## 8. Inspect representative reviews from K = 6

### Question

Do real reviews support the themes suggested by the top keywords?

### What we did

We inspect random examples from each cluster rather than only the first rows in the dataset.

In [24]:
# Inspect five random reviews from each K=6 cluster.
for cluster_number in range(6):

    print(f"\n{'=' * 70}")
    print(f"CLUSTER {cluster_number}")
    print(f"{'=' * 70}")

    # Select all reviews belonging to this cluster.
    cluster_reviews = df[
        df["cluster"] == cluster_number
    ]

    # Randomly select five examples.
    # random_state makes the sample reproducible.
    sample_reviews = cluster_reviews.sample(
        n=5,
        random_state=42
    )

    # Display the review text.
    display(
        sample_reviews[
            ["reviews.text"]
        ]
    )


CLUSTER 0


,reviews.text
18272,I've previously owned a Nook and Ipad. This eR...
229,"Great for reading - nice image, easy on the ey..."
20090,"Small, compact and perfect for travel. Able to..."
21185,Really like to read ebooks on the Kindle. The ...
19782,I love this Kindle. It is a replacement for my...



CLUSTER 1


,reviews.text
12694,This tablet is easy to use. It is also durable...
5401,This tablet is a great price. Perfect for thos...
21888,love it... easy to use and efficient...appropr...
3729,"I already have a Kindle Paperwhite, which I do..."
29478,I have an Amazon Echo and the Amazon Tap works...



CLUSTER 2


...

,reviews.text
15655,Nothing is perfect (even IPad). But for what y...
11672,Will Suggest this tablet to my friends. Yes I ...
30039,The Fire Tv is fast and reliable. With the eas...
24682,"I like this product, the sound and quality is ..."
27526,Great to interact with a product by voice that...



CLUSTER 4


,reviews.text
18807,Got this as a birthday gift for my wife..she l...
21916,This was a birthday gift for my son-in-law and...
2745,My grandson loves it! It's exactly what he nee...
11307,We purchased the Kindle Fire as a Christmas gi...
21646,My husband got this for Father's Day. He loves...



CLUSTER 5


,reviews.text
14006,My grandchildren love this. Its the perfit siz...
559,Great inexpensive tablet for everyday use. It'...
526,Much harder to work than an android. None of t...
21331,The tablet periodically shuts off for no reaso...
7618,I purchased this tablet mainly because of the ...


## 9. Select the final number of clusters

### Result

We compared KMeans solutions with K = 4, 5 and 6.

Silhouette Scores:

- K = 4: 0.0054
- K = 5: 0.0066
- K = 6: 0.0074

The absolute Silhouette Scores are low, which suggests that customer-review topics overlap considerably.

However, K = 6 achieved the highest Silhouette Score and also produced the most interpretable topic separation during qualitative inspection.

### Decision / Finding

We select **K = 6** as the final clustering configuration.

The decision is based on both:

- quantitative evidence: highest Silhouette Score among the tested values
- qualitative evidence: clearer and more interpretable review themes

## 10. Characterize and name the final clusters

### Question

What does each of the six final clusters represent?

### What we did

We combine three types of evidence:

- top TF-IDF keywords
- real review examples
- existing metadata such as sentiment and rating

The human-readable names are interpretations added after clustering. They were not used by KMeans during training.

In [25]:
# Connect each numerical cluster ID
# to a human-readable interpretation.
cluster_names = {
    0: "Kindle / Reading",
    1: "Ease of Use / Setup",
    2: "Amazon Devices / Mixed Usage",
    3: "Product Satisfaction / Recommendations",
    4: "Gifts / Family",
    5: "Tablets / Value & Everyday Use"
}

# Add the readable cluster name to every review.
df["cluster_name"] = df["cluster"].map(cluster_names)

# Display how many reviews belong to each named cluster.
df["cluster_name"].value_counts()

cluster_name
Amazon Devices / Mixed Usage              15435
Tablets / Value & Everyday Use             5842
Kindle / Reading                           4679
Gifts / Family                             3749
Ease of Use / Setup                        2706
Product Satisfaction / Recommendations     2204
Name: count, dtype: int64

In [26]:
# Count how many reviews belong to each cluster.
cluster_sizes = (
    df.groupby(["cluster", "cluster_name"])
    .size()
    .reset_index(name="review_count")
)

# Calculate what percentage of the full dataset
# belongs to each cluster.
cluster_sizes["percentage"] = (
    cluster_sizes["review_count"]
    / len(df)
    * 100
).round(2)

cluster_sizes

,cluster,cluster_name,review_count,percentage
0,0,Kindle / Reading,4679,13.52
1,1,Ease of Use / Setup,2706,7.82
2,2,Amazon Devices / Mixed Usage,15435,44.59
3,3,Product Satisfaction / Recommendations,2204,6.37
4,4,Gifts / Family,3749,10.83
5,5,Tablets / Value & Everyday Use,5842,16.88


In [27]:
# Build the output path for the clustered reviews.
CLUSTERED_DATA_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "clustered_reviews.csv"
)

# Save all reviews together with their final
# numerical cluster and readable cluster name.
df.to_csv(
    CLUSTERED_DATA_PATH,
    index=False
)

print(
    "Clustered dataset saved to:",
    CLUSTERED_DATA_PATH
)

Clustered dataset saved to: /Users/caiomaia/ironhack-workspace/project-nlp-customer-reviews-df-cm/datasets/clustered_reviews.csv


In [28]:
# Reload the saved file to confirm
# that the clustering results were saved correctly.
verification_df = pd.read_csv(
    CLUSTERED_DATA_PATH,
    low_memory=False
)

print("Saved shape:", verification_df.shape)

print(
    "Has cluster column:",
    "cluster" in verification_df.columns
)

print(
    "Has cluster_name column:",
    "cluster_name" in verification_df.columns
)

Saved shape: (34615, 9)
Has cluster column: True
Has cluster_name column: True


### Result

K = 6 produced six interpretable clusters.

The largest cluster, **Amazon Devices / Mixed Usage**, contains 44.59% of all reviews, which indicates that a substantial portion of the dataset still uses broad or overlapping product language.

The remaining clusters are more specific and capture clearer themes such as:

- Kindle / Reading
- Ease of Use / Setup
- Gifts / Family
- Tablets / Value & Everyday Use
- Product Satisfaction / Recommendations

### Limitation

The clustering is useful for discovering broad themes, but the low Silhouette Scores and the large mixed-use cluster show that review topics overlap substantially.

Cluster names are human interpretations based on keywords and sample reviews, not labels learned directly by KMeans.

In [29]:
# Compare sentiment percentages inside each cluster.
cluster_sentiment = pd.crosstab(
    df["cluster_name"],
    df["sentiment"],
    normalize="index"
)

# Convert proportions into percentages.
cluster_sentiment = (
    cluster_sentiment
    .mul(100)
    .round(2)
)

cluster_sentiment

sentiment,negative,neutral,positive
cluster_name,,,
Amazon Devices / Mixed Usage,3.23,4.89,91.88
Ease of Use / Setup,0.30,0.78,98.93
Gifts / Family,0.93,2.64,96.43
Kindle / Reading,2.01,3.06,94.93
Product Satisfaction / Recommendations,1.50,3.13,95.37
Tablets / Value & Everyday Use,2.43,7.05,90.52


In [30]:
# Calculate the average rating inside each cluster.
cluster_ratings = (
    df.groupby("cluster_name")["reviews.rating"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

cluster_ratings

cluster_name
Ease of Use / Setup                       4.75
Gifts / Family                            4.68
Kindle / Reading                          4.67
Product Satisfaction / Recommendations    4.65
Amazon Devices / Mixed Usage              4.55
Tablets / Value & Everyday Use            4.45
Name: reviews.rating, dtype: float64

### Result

The clusters show different sentiment and rating patterns even though sentiment was not used to create them.

Key observations:

- **Ease of Use / Setup** has the highest average rating (4.75) and the highest Positive share (98.93%).
- **Gifts / Family** and **Kindle / Reading** are also strongly positive.
- **Amazon Devices / Mixed Usage** is more mixed than the more specific clusters.
- **Tablets / Value & Everyday Use** has the lowest average rating (4.45) and the highest Neutral share (7.05%).

### Decision / Finding

The metadata supports the qualitative cluster interpretations.

K = 6 produces meaningful thematic groups with distinct review patterns, although the overall dataset remains strongly biased toward Positive reviews.

In [16]:
# Create a sentiment-percentage table indexed by cluster ID.
sentiment_summary = pd.crosstab(
    df["cluster"],
    df["sentiment"],
    normalize="index"
).mul(100)

# Calculate average rating for each cluster ID.
rating_summary = (
    df.groupby("cluster")["reviews.rating"]
    .mean()
)

# Start from the cluster size table we already created.
cluster_summary = cluster_sizes.copy()

# Add the average rating for each cluster.
cluster_summary["average_rating"] = (
    cluster_summary["cluster"]
    .map(rating_summary)
    .round(2)
)

# Add Negative sentiment percentage.
cluster_summary["negative_%"] = (
    cluster_summary["cluster"]
    .map(sentiment_summary["negative"])
    .round(2)
)

# Add Neutral sentiment percentage.
cluster_summary["neutral_%"] = (
    cluster_summary["cluster"]
    .map(sentiment_summary["neutral"])
    .round(2)
)

# Add Positive sentiment percentage.
cluster_summary["positive_%"] = (
    cluster_summary["cluster"]
    .map(sentiment_summary["positive"])
    .round(2)
)

# Display the final cluster characterization.
cluster_summary

,cluster,cluster_name,review_count,percentage,average_rating,negative_%,neutral_%,positive_%
0,0,Kindle / Reading,4679,13.52,4.67,2.01,3.06,94.93
1,1,Ease of Use / Setup,2706,7.82,4.75,0.30,0.78,98.93
2,2,Amazon Devices / Mixed Usage,15435,44.59,4.55,3.23,4.89,91.88
3,3,Product Satisfaction / Recommendations,2204,6.37,4.65,1.50,3.13,95.37
4,4,Gifts / Family,3749,10.83,4.68,0.93,2.64,96.43
5,5,Tablets / Value & Everyday Use,5842,16.88,4.45,2.43,7.05,90.52


## Final Task 2 Findings

We tested KMeans clustering with K = 4, 5 and 6 using TF-IDF features from `reviews.text`.

K = 6 was selected because it achieved the highest Silhouette Score among the tested values and produced the most interpretable review themes.

Final clusters:

- Kindle / Reading
- Ease of Use / Setup
- Amazon Devices / Mixed Usage
- Product Satisfaction / Recommendations
- Gifts / Family
- Tablets / Value & Everyday Use

The clusters also show different sentiment and rating patterns.

For example, Ease of Use / Setup has the highest average rating and Positive share, while Tablets / Value & Everyday Use has the lowest average rating and the highest Neutral share.

A limitation is that Amazon Devices / Mixed Usage still contains 44.59% of the dataset, indicating substantial overlap between review topics.

**Selected clustering configuration: TF-IDF + KMeans with K = 6.**

## Final Task 2 Findings

We tested KMeans clustering with K = 4, 5 and 6 using TF-IDF features from `reviews.text`.

K = 6 was selected because it achieved the highest Silhouette Score among the tested values and produced the most interpretable review themes.

Final clusters:

- Kindle / Reading
- Ease of Use / Setup
- Amazon Devices / Mixed Usage
- Product Satisfaction / Recommendations
- Gifts / Family
- Tablets / Value & Everyday Use

The clusters also show different sentiment and rating patterns.

For example, Ease of Use / Setup has the highest average rating and Positive share, while Tablets / Value & Everyday Use has the lowest average rating and the highest Neutral share.

A limitation is that Amazon Devices / Mixed Usage still contains 44.59% of the dataset, indicating substantial overlap between review topics.

**Selected clustering configuration: TF-IDF + KMeans with K = 6.**

In [18]:
# Reload the saved file to confirm
# the clustering results were saved correctly.
verification_df = pd.read_csv(
    CLUSTERED_DATA_PATH,
    low_memory=False
)

print("Saved shape:", verification_df.shape)

print(
    "Has cluster column:",
    "cluster" in verification_df.columns
)

print(
    "Has cluster_name column:",
    "cluster_name" in verification_df.columns
)

Saved shape: (34615, 9)
Has cluster column: True
Has cluster_name column: True


### Naming note

The six-cluster solution is final.

Four cluster themes are clearly aligned between both team analyses:

- Kindle / Reading
- Ease of Use / Setup
- Kids / Family / Gifts
- Tablets / Value

Two broader clusters are still being aligned between team members before final naming.

Current provisional names:
- Amazon Devices / Mixed Usage
- Product Satisfaction / Recommendations